
# Control MC Off-Policy con Muestreo por Importancia Ponderado
### GridWorld 2×3 — Eemplo de las diapositivas

Este cuaderno implementa **paso a paso** el algoritmo de *Monte Carlo Off-Policy Control*
con muestreo por importancia **ponderado** (weighted importance sampling), aplicado a un
GridWorld de 2 filas × 3 columnas.

```
(0,0)  (0,1)  (0,2)
(1,0)  (1,1)  (1,2)
```

| Elemento | Celda | Recompensa | Tipo |
|---|---|---|---|
| Inicio 🤖 | (0,0) | — | — |
| Fuego 🔥 | (1,0) | **−5** | Terminal |
| Meta / Batería 🔋 | (1,2) | **+10** | Terminal |

- **γ = 1**
- **μ (política de comportamiento):** en cada estado, elige entre sus dos acciones válidas
  con probabilidad 0.5 cada una (política blanda, fija durante todo el entrenamiento).
- **π (política objetivo):** determinista, se actualiza en cada paso como
  `π(St) ← argmax_a Q(St, a)`.

**Pseudocódigo implementado:**

```
Initialize, for all s ∈ S, a ∈ A(s):
    Q(s,a) ← arbitrary
    C(s,a) ← 0
    π(s)   ← a deterministic policy greedy with respect to Q
Repeat episode = 10:
    Generate an episode using any soft policy μ:  S0,A0,R1,...,ST-1,AT-1,RT,ST
    G ← 0
    W ← 1
    For t = T-1, T-2, ..., 0:
        G ← γG + R(t+1)
        C(St,At) ← C(St,At) + W
        Q(St,At) ← Q(St,At) + W/C(St,At) [G − Q(St,At)]
        π(St) ← argmax_a Q(St,a)
        W ← W · π(At|St) / μ(At|St)
        If W = 0 then ExitForLoop
```

Usa los **3 botones** del panel de control para avanzar la simulación:

- **▶ Paso** — ejecuta un único paso atómico (un movimiento del robot durante la
  generación del episodio, o una actualización de `t` durante el aprendizaje hacia atrás).
- **⏭ Episodio** — genera y aprende un episodio completo de una vez.
- **⏩ Completo** — corre los 10 episodios hasta el final y muestra la curva de convergencia.


In [9]:

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

# ----------------------------- Entorno ------------------------------------
GAMMA = 1.0
ROWS, COLS = 2, 3
START = (0, 0)
FIRE = (1, 0)
GOAL = (1, 2)
N_EPISODES = 10
SEED = 7

ARROW = {"up": "↑", "down": "↓", "left": "←", "right": "→"}

# Acciones válidas por estado, exactamente como en la tabla de mu del enunciado
STATE_ACTIONS = {
    (0, 0): ["right", "down"],
    (0, 1): ["right", "up"],
    (0, 2): ["down", "up"],
    (1, 1): ["right", "up"],
}
NONTERMINAL_STATES = list(STATE_ACTIONS.keys())


def mu_prob(s, a):
    """mu(a|s): todas las acciones válidas de s tienen igual probabilidad (0.5)."""
    return 1.0 / len(STATE_ACTIONS[s])


def env_step(s, a):
    """Transición determinista. Un movimiento inválido (fuera de la grilla)
    hace que el agente 'choque' y se quede en la misma celda."""
    r, c = s
    nr, nc = r, c
    if a == "right":
        nc = c + 1
    elif a == "left":
        nc = c - 1
    elif a == "down":
        nr = r + 1
    elif a == "up":
        nr = r - 1
    if not (0 <= nr < ROWS and 0 <= nc < COLS):
        nr, nc = r, c
    ns = (nr, nc)
    reward, done = 0.0, False
    if ns == FIRE:
        reward, done = -5.0, True
    elif ns == GOAL:
        reward, done = 10.0, True
    return ns, reward, done


print("Entorno definido: GridWorld 2x3 | Inicio", START, "| Fuego", FIRE, "| Meta", GOAL)


Entorno definido: GridWorld 2x3 | Inicio (0, 0) | Fuego (1, 0) | Meta (1, 2)


In [10]:

# ---------------------- Agente MC Off-Policy (WIS) -------------------------
class MCOffPolicyAgent:
    """Implementa el algoritmo como una máquina de estados para poder
    avanzarlo 'por paso', 'por episodio' o 'completo'."""

    def __init__(self, seed=SEED):
        self.rng = random.Random(seed)

        self.Q = {(s, a): 0.0 for s in STATE_ACTIONS for a in STATE_ACTIONS[s]}
        self.C = {(s, a): 0.0 for s in STATE_ACTIONS for a in STATE_ACTIONS[s]}
        # pi inicial = la de la tabla de la diapositiva (↓, ↑, ↑, ↑)
        self.pi = {(0, 0): "down", (0, 1): "up", (0, 2): "up", (1, 1): "up"}

        self.episode_num = 0
        self.phase = "idle"          # idle -> gen -> learn -> idle/done
        self.traj = []
        self.gen_idx = 0
        self.robot_state = START
        self.learn_idx = None
        self.G = 0.0
        self.W = 1.0

        self.reward_history = []
        self.rows = []                # filas de la prueba de escritorio
        self.last_event = {"type": "init", "msg": "Agente inicializado."}

    # -- utilidades -----------------------------------------------------
    def choose_mu_action(self, s):
        acts = STATE_ACTIONS[s]
        probs = [mu_prob(s, a) for a in acts]
        return self.rng.choices(acts, weights=probs, k=1)[0]

    def generate_full_episode(self, max_steps=40):
        s = START
        traj = []
        for _ in range(max_steps):
            a = self.choose_mu_action(s)
            ns, r, done = env_step(s, a)
            traj.append((s, a, r))
            s = ns
            if done:
                break
        return traj

    def argmax_actions(self, s):
        acts = STATE_ACTIONS[s]
        qvals = [self.Q[(s, a)] for a in acts]
        mx = max(qvals)
        return [a for a, q in zip(acts, qvals) if q == mx]

    def update_pi(self, s):
        best = self.argmax_actions(s)
        if self.pi[s] not in best:
            self.pi[s] = best[0]
        return self.pi[s]

    @property
    def done(self):
        return self.episode_num >= N_EPISODES and self.phase == "idle"

    # -- máquina de estados: UN paso atómico -----------------------------
    def step(self):
        if self.done:
            self.last_event = {"type": "done", "msg": "Ya se completaron los 10 episodios."}
            return self.last_event

        if self.phase == "idle":
            self.traj = self.generate_full_episode()
            self.gen_idx = 0
            self.robot_state = START
            self.phase = "gen"
            self.episode_num += 1
            self.last_event = {
                "type": "episode_start",
                "episode": self.episode_num,
                "T": len(self.traj),
                "msg": f"Episodio {self.episode_num}: generado con μ, T={len(self.traj)} pasos.",
            }
            return self.last_event

        if self.phase == "gen":
            s, a, r = self.traj[self.gen_idx]
            ns, _, _ = env_step(s, a)
            self.robot_state = ns
            self.gen_idx += 1
            info = {"type": "move", "episode": self.episode_num, "from": s, "action": a, "to": ns}
            if self.gen_idx >= len(self.traj):
                self.phase = "learn"
                self.learn_idx = len(self.traj) - 1
                self.G, self.W = 0.0, 1.0
                info["msg"] = "Episodio generado por completo. Comienza el aprendizaje hacia atrás (backward pass)."
            else:
                info["msg"] = f"Robot se mueve {s} --{ARROW[a]}--> {ns}"
            self.last_event = info
            return info

        if self.phase == "learn":
            t = self.learn_idx
            St, At, Rt1 = self.traj[t]
            self.G = GAMMA * self.G + Rt1
            self.C[(St, At)] += self.W
            Q_old = self.Q[(St, At)]
            Q_new = Q_old + (self.W / self.C[(St, At)]) * (self.G - Q_old)
            self.Q[(St, At)] = Q_new
            pi_new = self.update_pi(St)

            row = {
                "Episodio": self.episode_num, "t": t, "St": str(St), "At": ARROW[At],
                "R(t+1)": Rt1, "G": round(self.G, 3), "W": round(self.W, 4),
                "C(St,At)": round(self.C[(St, At)], 3),
                "Q_old": round(Q_old, 3), "Q_new": round(Q_new, 3), "pi_new(St)": ARROW[pi_new],
            }
            self.rows.append(row)

            prob_pi = 1.0 if At == self.pi[St] else 0.0
            self.W = self.W * prob_pi / mu_prob(St, At)
            self.learn_idx -= 1

            episode_finished = (self.W == 0) or (self.learn_idx < 0)
            info = {"type": "row", "episode": self.episode_num, "row": row,
                    "msg": f"t={t}: St={St}, At={ARROW[At]}, W→{round(self.W,4)}"}

            if episode_finished:
                total_reward = sum(r for (_, _, r) in self.traj)
                self.reward_history.append(total_reward)
                self.phase = "idle"
                info["episode_done"] = True
                info["total_reward"] = total_reward
                info["msg"] += f"  |  Episodio {self.episode_num} terminado (W=0 o t<0). Retorno total = {total_reward}"
            self.last_event = info
            return info

    def run_episode(self, animate=None, delay=0.0):
        """Corre pasos hasta terminar (o comenzar) un episodio completo."""
        import time
        started = False
        while True:
            ev = self.step()
            started = True
            if animate is not None:
                animate(ev)
            if delay:
                time.sleep(delay)
            if ev.get("episode_done") or ev.get("type") == "done":
                break
        return ev

    def run_all(self, animate=None, delay=0.0):
        while not self.done:
            self.run_episode(animate=animate, delay=delay)

    def table_df(self):
        if not self.rows:
            return pd.DataFrame(columns=["Episodio", "t", "St", "At", "R(t+1)", "G", "W",
                                          "C(St,At)", "Q_old", "Q_new", "pi_new(St)"])
        return pd.DataFrame(self.rows)


print("Clase MCOffPolicyAgent lista.")


Clase MCOffPolicyAgent lista.


In [13]:

# ------------------------------- Dibujo -------------------------------
CELL_BG = "#1b232e"
PANEL_BG = "#141a22"
FIRE_BG = "#e2703a"
GOAL_BG = "#3ecf9e"
TEXT_COLOR = "#e7ecf2"
GRID_LINE = "#2e3a48"
ROBOT_COLOR = "#f0b429"
MU_COLOR = "#7c93b3"

plt.rcParams["figure.facecolor"] = PANEL_BG
plt.rcParams["axes.facecolor"] = PANEL_BG
plt.rcParams["text.color"] = TEXT_COLOR
plt.rcParams["font.family"] = "DejaVu Sans"


def _base_grid(ax, title):
    ax.set_xlim(0, COLS)
    ax.set_ylim(0, ROWS)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.set_title(title, color=TEXT_COLOR, fontsize=12, fontweight="bold", pad=10)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for r in range(ROWS):
        for c in range(COLS):
            face = CELL_BG
            if (r, c) == FIRE:
                face = FIRE_BG
            elif (r, c) == GOAL:
                face = GOAL_BG
            rect = mpatches.Rectangle((c, ROWS - 1 - r), 1, 1, facecolor=face,
                                       edgecolor=GRID_LINE, linewidth=1.5)
            ax.add_patch(rect)
            if (r, c) == FIRE:
                ax.text(c + 0.5, ROWS - 1 - r + 0.5, "FUEGO\n-5", ha="center", va="center",
                         color="white", fontsize=9, fontweight="bold")
            elif (r, c) == GOAL:
                ax.text(c + 0.5, ROWS - 1 - r + 0.5, "META\n+10", ha="center", va="center",
                         color="#0c2b22", fontsize=9, fontweight="bold")


def draw_mu_grid_static():
    """La grilla de mu es estática: se dibuja una sola vez."""
    fig, ax = plt.subplots(figsize=(3.6, 2.6))
    _base_grid(ax, "μ (política de comportamiento) — estática")
    for (r, c), acts in STATE_ACTIONS.items():
        label = "  ".join(f"{ARROW[a]} .5" for a in acts)
        ax.text(c + 0.5, ROWS - 1 - r + 0.5, label, ha="center", va="center",
                 color=MU_COLOR, fontsize=11, fontweight="bold")
    plt.tight_layout()
    return fig


def draw_pi_and_robot(agent):
    """Dos grillas dinámicas: pi (arrows) y el entorno con el robot."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 2.8))

    _base_grid(ax1, "π (política objetivo) — dinámica")
    for (r, c), _ in STATE_ACTIONS.items():
        ax1.text(c + 0.5, ROWS - 1 - r + 0.5, ARROW[agent.pi[(r, c)]], ha="center", va="center",
                  color="#f5f7fa", fontsize=20, fontweight="bold")

    _base_grid(ax2, f"Entorno — episodio {max(agent.episode_num,1)}")
    rr, rc = agent.robot_state
    #if (rr, rc) not in (FIRE, GOAL):
    #    circ = mpatches.Circle((rc + 0.5, ROWS - 1 - rr + 0.5), 0.32,
    #                            facecolor=ROBOT_COLOR, edgecolor="#7a4a12", linewidth=1.5, zorder=5)
    #    ax2.add_patch(circ)
    #    ax2.text(rc + 0.5, ROWS - 1 - rr + 0.5, "A", ha="center", va="center",
    #              color="#241300", fontsize=12, fontweight="bold", zorder=6)
    #else:
    #    ax2.text(rc + 0.5, ROWS - 1 - rr + 0.35, "🤖", ha="center", va="center", fontsize=13, zorder=6)

    rr, rc = agent.robot_state
    circ = mpatches.Circle((rc + 0.5, ROWS - 1 - rr + 0.5), 0.32,
                            facecolor=ROBOT_COLOR, edgecolor="#7a4a12", linewidth=1.5, zorder=5)
    ax2.add_patch(circ)
    ax2.text(rc + 0.5, ROWS - 1 - rr + 0.5, "R", ha="center", va="center",
              color="#241300", fontsize=12, fontweight="bold", zorder=6)

    plt.tight_layout()
    return fig


def draw_convergence(agent):
    fig, ax = plt.subplots(figsize=(6.4, 3.6))

    for spine in ax.spines.values():
        spine.set_color(TEXT_COLOR)
        spine.set_linewidth(1.0)
    ax.tick_params(colors=TEXT_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)

    eps = list(range(1, len(agent.reward_history) + 1))
    ax.plot(eps, agent.reward_history, marker="o", linewidth=2, color="#3ecf9e")
    ax.axhline(0, color="#888", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total (retorno G0)")
    ax.set_title("Curva de convergencia — recompensa por episodio")
    ax.set_xticks(eps)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


print("Funciones de dibujo listas.")


Funciones de dibujo listas.


In [14]:

# ----------------------------- Dashboard -------------------------------
agent = MCOffPolicyAgent(seed=SEED)

btn_step = widgets.Button(description="▶ Paso", button_style="info", layout=widgets.Layout(width="120px"))
btn_episode = widgets.Button(description="⏭ Episodio", button_style="warning", layout=widgets.Layout(width="120px"))
btn_all = widgets.Button(description="⏩ Completo", button_style="success", layout=widgets.Layout(width="120px"))
btn_reset = widgets.Button(description="↺ Reiniciar", button_style="danger", layout=widgets.Layout(width="120px"))
status_label = widgets.HTML(value="<b>Estado:</b> listo para comenzar.")

out_mu = widgets.Output()
out_dynamic = widgets.Output()
out_table = widgets.Output()
out_chart = widgets.Output()

with out_mu:
    fig_mu = draw_mu_grid_static()
    plt.show()
    plt.close(fig_mu)


def refresh_dynamic():
    with out_dynamic:
        clear_output(wait=True)
        fig = draw_pi_and_robot(agent)
        plt.show()
        plt.close(fig)


def refresh_table():
    with out_table:
        clear_output(wait=True)
        df = agent.table_df()
        display(df.tail(15).style.set_caption(
            f"Prueba de escritorio (mostrando últimas 15 de {len(df)} filas totales)"
        ))


def refresh_chart(force=False):
    if agent.done or force:
        with out_chart:
            clear_output(wait=True)
            if agent.reward_history:
                fig = draw_convergence(agent)
                plt.show()
                plt.close(fig)


def set_status(msg):
    status_label.value = f"<b>Estado:</b> {msg}"


def render_all():
    refresh_dynamic()
    refresh_table()
    refresh_chart()


def on_step(_):
    ev = agent.step()
    set_status(ev.get("msg", ""))
    render_all()


def on_episode(_):
    def animate(ev):
        set_status(ev.get("msg", ""))
        refresh_dynamic()
    agent.run_episode(animate=animate, delay=0.15)
    refresh_table()
    refresh_chart()


def on_all(_):
    def animate(ev):
        set_status(ev.get("msg", ""))
        refresh_dynamic()
    agent.run_all(animate=animate, delay=0.05)
    refresh_table()
    refresh_chart(force=True)
    set_status(f"Completado: {N_EPISODES} episodios. Revisa la curva de convergencia abajo.")


def on_reset(_):
    global agent
    agent = MCOffPolicyAgent(seed=SEED)
    set_status("Reiniciado. Listo para comenzar de nuevo.")
    with out_chart:
        clear_output(wait=True)
    render_all()


btn_step.on_click(on_step)
btn_episode.on_click(on_episode)
btn_all.on_click(on_all)
btn_reset.on_click(on_reset)

controls = widgets.HBox([btn_step, btn_episode, btn_all, btn_reset])
grids_row = widgets.HBox([out_mu, out_dynamic])

display(widgets.VBox([
    controls,
    status_label,
    grids_row,
    widgets.HTML("<hr><b>Prueba de escritorio</b> (t, W, G, C, Q_old, Q_new, π_new)"),
    out_table,
    widgets.HTML("<hr><b>Curva de convergencia</b> (aparece al terminar el episodio 10, o con '⏩ Completo')"),
    out_chart,
]))

render_all()



### Notas de lectura

- **`(1,1)` no se visita**: con las acciones disponibles definidas en la tabla de μ,
  el único camino desde `(0,0)` pasa por `(0,1) → (0,2) → META`, o directo a `(0,0) → FUEGO`.

- **Muestreo por importancia ponderado**: como π es determinista, apenas la acción tomada por
  μ difiere de la acción greedy de π, el ratio `π(At|St)` se vuelve 0 y el peso `W` colapsa a 0,
  cortando el resto del backward pass (`ExitForLoop`). Por eso muchas filas de la tabla
  corresponden sólo a los últimos pasos de cada episodio.

- **Convergencia**: a medida que π se vuelve greedy respecto a Q, la política debería
  estabilizarse en `(0,0)→→, (0,1)→→, (0,2)→↓`, el camino más corto hacia la batería,
  evitando el fuego.

Para volver a ejecutar la demo desde cero, usar el botón **↺ Reiniciar** o volver a correr
la celda del *Dashboard*.
